# Fraud Detection — LightGBM Training

Trains a LightGBM binary classifier on the **point-in-time feature table** produced by
`airflow/etl/feature_etl.py` (`application.transaction_features`), joined to ground-truth
labels in `application.labels`. No MLflow — artifacts are written to `models/`.

**Design notes**
- Raw entity IDs (`user_id`, `card_id`, ...) and `created_at` are dropped from the feature
  set: entity behaviour is already encoded by the velocity / graph features, and raw IDs
  invite identity overfitting.
- `is_declined` is **excluded** — it is the current transaction's post-authorization status,
  which is not available to a pre-auth scorer (the ETL flags it as leakage-prone). The
  leakage-safe historical version `card_declines_24h` is kept.
- Split is **temporal** (train on the earlier rows, validate on the most recent) to mirror
  production, where the model always scores the future.

In [19]:
import json
import os
import time
import warnings
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
import psycopg
from sklearn.metrics import average_precision_score, roc_auc_score

In [20]:
CURRENT_DIR = Path().cwd()
# When run from scripts/training/, hop up to the repo root; else assume cwd is the root.
REPO_ROOT = CURRENT_DIR.parents[1] if CURRENT_DIR.name == "training" else CURRENT_DIR
MODEL_DIR = REPO_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)


def load_env_file(path: Path) -> None:
    if not path.exists():
        return
    for raw_line in path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


load_env_file(REPO_ROOT / ".env")

CONNINFO = (
    f"host={os.getenv('POSTGRES_HOST')} port={os.getenv('POSTGRES_PORT')} "
    f"user={os.getenv('POSTGRES_USER')} password={os.getenv('POSTGRES_PASSWORD')} "
    f"dbname={os.getenv('POSTGRES_DB')}"
)
print("Model artifacts ->", MODEL_DIR)

Model artifacts -> /home/lehoangvu/fraud-detection/models


In [21]:
SEED = 42
TARGET = "label"

# Columns present in the feature table that are NOT model inputs.
ID_COLS = ["transaction_id", "user_id", "card_id", "merchant_id", "device_id"]
DROP_COLS = set(ID_COLS) | {
    "created_at",      # split key only, not a feature
    "computed_at",     # ETL bookkeeping
    "label",           # target
    "label_source",    # metadata
    "is_declined",     # post-auth leakage — see notebook header
}

# Low-cardinality text/boolean columns -> label-encoded and passed as LightGBM categoricals.
CATEGORICAL_COLS = [
    "channel", "card_brand", "card_type",
    "customer_segment", "merchant_category",
    "is_virtual", "email_verified",
]

LGB_PARAMS = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "learning_rate": 0.02,
    "num_leaves": 128,
    "min_child_samples": 100,
    "feature_fraction": 0.7,
    "bagging_fraction": 0.85,
    "bagging_freq": 1,
    "lambda_l1": 0.1,
    "lambda_l2": 0.1,
    "verbose": -1,
    "n_jobs": -1,
    "seed": SEED,
}
NUM_BOOST_ROUND = 5000
EARLY_STOPPING = 200
VAL_FRAC = 0.15          # most-recent slice held out for early stopping / eval
FINAL_ROUND_BUMP = 1.10  # phase B trains a bit past the early-stopped optimum


def log(msg: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

In [22]:
SOURCE_SQL = """
SELECT f.*, l.label
FROM application.transaction_features f
JOIN application.labels l ON l.transaction_id = f.transaction_id
WHERE l.label IS NOT NULL
ORDER BY f.created_at
"""


def load_data() -> pd.DataFrame:
    log("Loading labeled feature rows from Postgres ...")
    with psycopg.connect(CONNINFO) as conn:
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", category=UserWarning, module="pandas")
            df = pd.read_sql(SOURCE_SQL, conn)
    df["created_at"] = pd.to_datetime(df["created_at"], utc=True)
    df[TARGET] = df[TARGET].astype(np.int8)
    pos = int(df[TARGET].sum())
    log(f"  rows: {len(df):,}  positives: {pos:,} ({pos / len(df):.3%})")
    return df.sort_values("created_at").reset_index(drop=True)


df = load_data()
df.head()

[15:28:28] Loading labeled feature rows from Postgres ...


/tmp/ipykernel_354040/28904050.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(SOURCE_SQL, conn)


[15:28:31]   rows: 238,146  positives: 1,174 (0.493%)


,transaction_id,created_at,user_id,card_id,merchant_id,device_id,amount_usd,log_amount,hour,weekday,...,card_amount_sum_24h,card_seconds_since_last_tx,card_amount_zscore,card_tx_seq,card_declines_24h,user_tx_count_24h,user_amount_sum_24h,user_seconds_since_last_tx,computed_at,label
0,61a81abb1873410d874b1eddcbcf3143,2024-09-11 20:20:37.168382+00:00,ebf73ab882dc405e8e143764f9254673,35e1ce212a104700977a263b1e5f0103,330004b812244505ba081a236c5f0944,f3625d5755a84afcbb531fb1a7720ce8,216.08,5.380266,20,2,...,216.08,NaN,NaN,1,0,1,216.08,NaN,2026-07-03 08:13:02.821590+00:00,0
1,7b0473a99a78485bac506a0b16a366f9,2024-09-25 00:38:46.224210+00:00,0796017f910249f0b0c33557d4bc03ec,c21b553d5b7b4838a83ad7465674979c,a3b67e74913445bdb0c83d39c7bca429,9f5d7feef73741378b928f0c0f831789,13.93,2.703373,0,2,...,13.93,NaN,NaN,1,0,1,13.93,NaN,2026-07-03 08:13:02.821590+00:00,0
2,629845e5c8094432946526c19ccb55c6,2024-09-25 17:23:57.447849+00:00,eb2b36f00355403184b692fd9f7f1bc8,ac49c0f887724e848d95a9886e9fefdc,49c8b8e0a5e14c96a44228c2a60ffa6a,037ce77f7eda4134bb80ccc6ac3ef76c,22.05,3.137666,17,2,...,22.05,NaN,NaN,1,0,1,22.05,NaN,2026-07-03 08:13:02.821590+00:00,0
3,bda933a46eaa4500b07330a541e9b6b4,2024-10-28 21:41:20.615168+00:00,a9d06cfb59a64d4fb3aeb15797dd6e78,259130037aa042b78cf8d0494fd06019,096857df5c024f6f8cdd4b68120259f1,34ac8cb0cda240f1acafb529aa131892,124.07,4.828874,21,0,...,124.07,NaN,NaN,1,0,1,124.07,NaN,2026-07-03 08:13:02.821590+00:00,0
4,e485e8cce8454487b3bf54988ed6ea6d,2024-11-09 18:44:23.565101+00:00,bb6bbb8167ff425a9c813a97199626f0,734cfa84c02b43019c522328ff9f17c3,8d98002e062a4683976521d729eac0ff,560c6c6b95cb462d8f6322279cca9b5b,8.00,2.197225,18,5,...,8.00,NaN,NaN,1,0,1,8.00,NaN,2026-07-03 08:13:02.821590+00:00,0


In [23]:
def encode_categoricals(df: pd.DataFrame) -> tuple[list[str], dict[str, dict[str, int]]]:
    """Label-encode the declared categorical columns; return (cat_cols, encoders).

    Booleans and text are stringified first so NULL/None maps to an explicit
    'missing' bucket and encodings are stable across runs.
    """
    encoders: dict[str, dict[str, int]] = {}
    cat_cols: list[str] = []
    for col in CATEGORICAL_COLS:
        if col not in df.columns:
            continue
        s = df[col].astype("string").fillna("missing").astype(str)
        mapping = {v: i for i, v in enumerate(sorted(s.unique()))}
        df[col] = s.map(mapping).astype(np.int32)
        encoders[col] = mapping
        cat_cols.append(col)
    log(f"Label-encoded {len(cat_cols)} categorical columns.")
    return cat_cols, encoders


def feature_columns(df: pd.DataFrame) -> list[str]:
    return [c for c in df.columns if c not in DROP_COLS]


cat_cols, encoders = encode_categoricals(df)
feat_cols = feature_columns(df)
log(f"{len(feat_cols)} feature columns  ({len(cat_cols)} categorical)")
feat_cols

[15:28:31] Label-encoded 7 categorical columns.
[15:28:31] 29 feature columns  (7 categorical)


['amount_usd',
 'log_amount',
 'hour',
 'weekday',
 'is_night',
 'channel',
 'card_brand',
 'card_type',
 'is_virtual',
 'customer_segment',
 'kyc_level',
 'email_verified',
 'merchant_category',
 'merchant_risk_level',
 'account_age_days',
 'card_age_days',
 'geo_mismatch',
 'foreign_ip',
 'recipient_differs',
 'card_tx_count_1h',
 'card_tx_count_24h',
 'card_amount_sum_24h',
 'card_seconds_since_last_tx',
 'card_amount_zscore',
 'card_tx_seq',
 'card_declines_24h',
 'user_tx_count_24h',
 'user_amount_sum_24h',
 'user_seconds_since_last_tx']

In [24]:
def temporal_split(df: pd.DataFrame, val_frac: float = VAL_FRAC):
    """Chronological split: earliest (1-val_frac) train, most-recent val_frac for eval."""
    n = len(df)
    cut = int(n * (1.0 - val_frac))
    idx = np.arange(n)
    tr, va = idx[:cut], idx[cut:]  # df is already sorted by created_at
    log(f"Split — train: {len(tr):,}  val: {len(va):,} (most-recent {val_frac:.0%})")
    return tr, va


def precision_at_k(y_true: np.ndarray, y_score: np.ndarray, k_frac: float) -> float:
    k = max(1, int(len(y_score) * k_frac))
    top = np.argpartition(-y_score, k - 1)[:k]
    return float(y_true[top].sum()) / k

In [25]:
def train_phase_a(df: pd.DataFrame, feat_cols: list[str], cat_cols: list[str]):
    """Fit on the earlier slice, early-stop on the most-recent slice; return (best_iter, metrics)."""
    log(f"Phase A - fit on first {1 - VAL_FRAC:.0%}, early-stop on last {VAL_FRAC:.0%}.")
    X, y = df[feat_cols], df[TARGET].to_numpy()
    tr, va = temporal_split(df)
    cat_idx = [c for c in cat_cols if c in feat_cols]

    pos = int(y[tr].sum())
    pos_weight = (len(tr) - pos) / max(1, pos)
    params = {**LGB_PARAMS, "scale_pos_weight": pos_weight}

    dtr = lgb.Dataset(X.iloc[tr], label=y[tr], categorical_feature=cat_idx)
    dva = lgb.Dataset(X.iloc[va], label=y[va], categorical_feature=cat_idx)
    booster = lgb.train(
        params, dtr,
        num_boost_round=NUM_BOOST_ROUND,
        valid_sets=[dva], valid_names=["valid"],
        callbacks=[lgb.early_stopping(EARLY_STOPPING), lgb.log_evaluation(period=200)],
    )
    val_pred = booster.predict(X.iloc[va], num_iteration=booster.best_iteration)
    metrics = {
        "val_roc_auc": float(roc_auc_score(y[va], val_pred)),
        "val_pr_auc": float(average_precision_score(y[va], val_pred)),
        "val_precision_at_1pct": precision_at_k(y[va], val_pred, 0.01),
        "val_precision_at_0.5pct": precision_at_k(y[va], val_pred, 0.005),
        "best_iteration": int(booster.best_iteration),
        "n_features": len(feat_cols),
        "n_train_phase_a": int(len(tr)),
        "scale_pos_weight": float(pos_weight),
    }
    log("Phase A metrics:\n" + json.dumps(metrics, indent=2))
    return booster.best_iteration, metrics


def train_phase_b(df: pd.DataFrame, feat_cols: list[str], cat_cols: list[str], best_iter: int):
    """Refit on the FULL labeled set for best_iter x bump rounds; save model. Return (booster, n_rounds)."""
    n_rounds = max(50, int(best_iter * FINAL_ROUND_BUMP))
    log(f"Phase B - refit on FULL labeled set ({len(df):,} rows) for {n_rounds} rounds.")
    X, y = df[feat_cols], df[TARGET].to_numpy()
    cat_idx = [c for c in cat_cols if c in feat_cols]

    pos = int(y.sum())
    params = {**LGB_PARAMS, "scale_pos_weight": (len(y) - pos) / max(1, pos)}
    dtr = lgb.Dataset(X, label=y, categorical_feature=cat_idx)
    booster = lgb.train(
        params, dtr,
        num_boost_round=n_rounds,
        callbacks=[lgb.log_evaluation(period=200)],
    )
    booster.save_model(str(MODEL_DIR / "model.bst"))
    return booster, n_rounds

In [26]:
best_iter, val_metrics = train_phase_a(df, feat_cols, cat_cols)
model, n_rounds = train_phase_b(df, feat_cols, cat_cols, best_iter)

metrics = {**val_metrics, "phase_b_n_rounds": n_rounds, "n_train_phase_b": int(len(df))}

[15:28:31] Phase A - fit on first 85%, early-stop on last 15%.
[15:28:31] Split — train: 202,424  val: 35,722 (most-recent 15%)
Training until validation scores don't improve for 200 rounds
[200]	valid's auc: 0.947887
[400]	valid's auc: 0.949441
[600]	valid's auc: 0.951006
[800]	valid's auc: 0.951637
[1000]	valid's auc: 0.952025
[1200]	valid's auc: 0.952068
Early stopping, best iteration is:
[1039]	valid's auc: 0.952499
[15:29:37] Phase A metrics:
{
  "val_roc_auc": 0.9524989147517161,
  "val_pr_auc": 0.8532173081818413,
  "val_precision_at_1pct": 0.4789915966386555,
  "val_precision_at_0.5pct": 0.9213483146067416,
  "best_iteration": 1039,
  "n_features": 29,
  "n_train_phase_a": 202424,
  "scale_pos_weight": 205.55510204081634
}
[15:29:37] Phase B - refit on FULL labeled set (238,146 rows) for 1142 rounds.


In [27]:
schema = {
    "version": 1,
    "source_table": "application.transaction_features",
    "label_table": "application.labels",
    "target": TARGET,
    "feature_columns": feat_cols,
    "categorical_features": cat_cols,
    "categorical_encoders": encoders,
    "dropped_columns": sorted(DROP_COLS),
    "lgb_params": LGB_PARAMS,
}

(MODEL_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))
(MODEL_DIR / "feature_schema.json").write_text(json.dumps(schema, indent=2))

log(f"Saved model + metrics + schema to {MODEL_DIR}")
print(json.dumps(metrics, indent=2))

[15:30:47] Saved model + metrics + schema to /home/lehoangvu/fraud-detection/models
{
  "val_roc_auc": 0.9524989147517161,
  "val_pr_auc": 0.8532173081818413,
  "val_precision_at_1pct": 0.4789915966386555,
  "val_precision_at_0.5pct": 0.9213483146067416,
  "best_iteration": 1039,
  "n_features": 29,
  "n_train_phase_a": 202424,
  "scale_pos_weight": 205.55510204081634,
  "phase_b_n_rounds": 1142,
  "n_train_phase_b": 238146
}
